# Data analysis

Every number in `docs/data_traps.md` should come from a cell in this notebook.

In [1]:
from pathlib import Path
import sys, json, re, csv, collections, ipaddress, warnings, zipfile, hashlib
import xml.etree.ElementTree as ET

ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from analysis.load import (DATA, load_alerts, load_sample_alerts, load_log,
                           load_inventory, load_intel, load_context, log_sources)
from analysis.helpers import (leaves, raw_records, encoded, exact_mask,
                              entity_values, change_times, change_window,
                              approval_refs, indexed_source, context_for, RUNBOOK_SOURCES)

def show(title, value):
    print(title)
    if isinstance(value, pd.Series):
        value = value.rename_axis("value").reset_index(name="count")
    if isinstance(value, pd.DataFrame):
        print(value.to_string(index=False, max_rows=50, max_colwidth=130))
    else:
        print(value)
    print()

# Tables below contain IDs/counts/controlled descriptions, never raw attacker prose.
# All joins and scans are local; no LLM, external indicator lookup or network I/O.

## 1. Overview

How many alerts are there by rule, product, severity and day? Which rules dominate? Are there any answer labels?

Affects: data/eval

In [2]:
alerts = load_alerts()
sample = load_sample_alerts()
logs = {source: load_log(source) for source in log_sources()}
inventory_names = sorted(p.stem for p in (DATA / "02_inventario").glob("*.csv"))
inventory = {name: load_inventory(name) for name in inventory_names}
assets, users = inventory["activos"], inventory["usuarios"]
changes, hr = inventory["calendario_cambios"], inventory["eventos_rrhh"]
services = inventory["servicios_negocio"]
catalog = load_context("catalogo_casos_de_uso")
intel = {name: load_intel(name) for name in ["ti_feed", "geoip_asn", "sandbox_informes"]}
show("Dataset counts", pd.DataFrame([
    ["alerts", len(alerts)], ["sample", len(sample)],
    ["log sources", len(logs)], ["log events", sum(map(len, logs.values()))],
    ["assets", len(assets)], ["users", len(users)], ["catalog rules", len(catalog)]
], columns=["table", "rows"]))
rule_counts = alerts.groupby(["rule_id", "source_product", "severity"]).size().reset_index(name="alerts")
show("All observed rules (rule severity, not final severity)", rule_counts.sort_values(["alerts", "rule_id"], ascending=[False, True]))
loudest = alerts.rule_id.value_counts()
show("Share of alerts from the loudest rules", {"top_8_rules_alerts": int(loudest.head(8).sum()),
     "share": round(loudest.head(8).sum() / len(alerts), 3)})
show("Products", alerts.source_product.value_counts())
show("Rule severities", alerts.severity.value_counts())
show("Alerts by UTC date", alerts.groupby(alerts.created_at_utc.dt.date).size())
label_fields = sorted({k for frame in [alerts, sample, *logs.values(), *inventory.values()]
                       for k in frame if re.search(r"ground.?truth|label|incident_id|verdict", k, re.I)})
show("Potential label fields (detections are not answer labels)", label_fields)
show("Catalog rules absent from alerts", sorted(set(catalog.rule_id) - set(alerts.rule_id)))
show("Exact alert/sample ID checks", {
    "duplicate_alert_ids": int(alerts.alert_id.duplicated().sum()),
    "duplicate_sample_ids": int(sample.alert_id.duplicated().sum()),
    "sample_ids_outside_full_set": len(set(sample.alert_id)-set(alerts.alert_id)),
})
full_by_id = {r["alert_id"]: r for r in raw_records(alerts)}
sample_differences = sum(r != full_by_id.get(r["alert_id"]) for r in raw_records(sample))
show("Sample row differences", sample_differences)

Dataset counts
        table   rows
       alerts  11061
       sample    150
  log sources     18
   log events 814471
       assets   1185
        users   1224
catalog rules     44

All observed rules (rule severity, not final severity)
   rule_id source_product severity  alerts
UC-WEB-001            WAF      low    3406
UC-EML-001       EMAIL_GW      low    2295
UC-EDR-002            EDR   medium    1349
UC-IDP-001            IDP   medium     901
UC-NET-001          PROXY   medium     746
UC-NET-005            DNS      low     561
UC-EDR-001            EDR     high     532
 UC-AI-001     AI_GATEWAY   medium     415
UC-EDR-003            EDR   medium     411
UC-NET-006            DNS   medium      91
UC-WEB-002            WAF     high      77
UC-IDP-002            IDP     high      62
UC-DLP-001            DLP     high      59
UC-CLD-003          CLOUD   medium      50
UC-NET-003           NGFW   medium      20
UC-WEB-003           ECOM     high      12
UC-DEV-001           CICD   me

In [3]:
# IDs are checked only to warn about them. Nothing below uses an ID number as a signal.
def id_breaks(ids):
    numbers = ids.str.extract(r"(\d+)$")[0].astype(int).sort_values()
    return int(numbers.diff().gt(1).sum())
show("Breaks in ID numbering per table", pd.DataFrame(
    [["alerts", len(alerts), id_breaks(alerts.alert_id)]] +
    [[source, len(d), id_breaks(d.event_id)] for source, d in logs.items()],
    columns=["table", "rows", "breaks"]))

Breaks in ID numbering per table
       table   rows  breaks
      alerts  11061       1
   ad_events    526       1
  ai_gateway   8817       1
 cicd_builds    559       1
  cloudtrail   6496       1
         dlp   1042       1
         dns 178221       1
ecom_account   7661       1
   ecom_auth  44064       1
  edr_events 112492       1
       email  56797       1
email_clicks  16249       1
    firewall  24648       1
   idp_audit    762       1
 idp_signins  76713       1
       proxy 194258       1
   sql_audit   3924       1
         vpn   9821       1
         waf  71421       1



**Observed:** Counts above are recomputed from files; detection verdicts in email, sandbox and AI logs describe particular controls, not ground truth for the alert. There is no supplied alert-level incident/verdict answer column. Eight rules produce 92% of alerts. IDs in all 19 tables have exactly one break in their numbering. The README's opening source count and the deck's runbook count should not be used as schema contracts.

**Design interpretation (data/eval, agent/prompt):** Keep raw rule severity and control verdicts separate from analyst labels. Build an independently reviewed evaluation set; do not treat the development sample as a scored test set. The prompt can say, in general terms, that a few rules produce most alerts and that a rule's severity isn't the final severity.

**Data leakage rule:** the ID breaks may reflect how the dataset was built, so they could act as an answer key. No code, prompt, rule, test selection, label or score may use ID numbers as a signal; IDs are only citation handles. This notebook reports that the breaks exist and doesn't say which records fall on either side. Nothing below selects by ID number.

## 2. Noise

For the loudest rules, group alerts by their key evidence fields. How many distinct patterns are there, and which alerts stand out?

Affects: agent/prompt, data/eval

In [4]:
# Selected evidence keys deliberately exclude message/event IDs and unique timestamps.
pattern_keys = {
 "UC-WEB-001": ["rule_id", "action"],
 "UC-EML-001": ["sender", "subject"],
 "UC-EDR-002": ["parent", "user", "decoded_preview"],
 "UC-IDP-001": ["device", "note"],
 "UC-NET-001": ["domain", "user_agent", "hosts_type"],
 "UC-NET-005": ["parent_domain"],
 "UC-EDR-001": ["process_tree", "signer", "edr_action"],
 "UC-AI-001": ["guardrail_category", "action"],
}
pattern_summary, pattern_examples = [], []
for rule in alerts.rule_id.value_counts().head(8).index:
    part = alerts[alerts.rule_id.eq(rule)].copy()
    keys = pattern_keys[rule]
    part["pattern"] = part.evidence.map(lambda e: encoded({k:e.get(k) for k in keys}))
    frequencies = part.pattern.value_counts()
    sizes = part.pattern.map(frequencies)
    pattern_summary.append([rule, len(part), len(frequencies), int(frequencies.iloc[0]),
                            int(sizes.eq(1).sum()), ", ".join(keys)])
    # Emit only IDs, never the pattern text, which may contain planted instructions.
    for _, r in part.assign(pattern_size=sizes).sort_values(["pattern_size", "alert_id"]).head(2).iterrows():
        pattern_examples.append([rule, r.alert_id, int(r.pattern_size)])
show("Largest rules: patterns under explicitly chosen fields", pd.DataFrame(pattern_summary,
    columns=["rule", "alerts", "patterns", "largest_pattern", "singleton_alerts", "group_fields"]))
show("Smallest-pattern leads (two per rule; rarity is not a verdict)", pd.DataFrame(
    pattern_examples, columns=["rule", "alert_id", "pattern_alerts"]))
show("Overall exact duplicate alert records (ignoring ID and creation time)",
     pd.Series([encoded({k:v for k,v in r.items() if k not in {"alert_id", "created_at"}})
                for r in raw_records(alerts)]).duplicated().sum())


Largest rules: patterns under explicitly chosen fields
      rule  alerts  patterns  largest_pattern  singleton_alerts                     group_fields
UC-WEB-001    3406         8              731                 0                  rule_id, action
UC-EML-001    2295         8              350                 0                  sender, subject
UC-EDR-002    1349         1             1349                 0    parent, user, decoded_preview
UC-IDP-001     901         3              899                 2                     device, note
UC-NET-001     746         7              161                 2   domain, user_agent, hosts_type
UC-NET-005     561         6              101                 0                    parent_domain
UC-EDR-001     532         2              531                 1 process_tree, signer, edr_action
 UC-AI-001     415         2              413                 0       guardrail_category, action

Smallest-pattern leads (two per rule; rarity is not a verdict)
      ru

**Observed:** Pattern counts depend on the displayed grouping fields. A changing subject or command can create a singleton; many different endpoints can share one pattern.

**Design interpretation (agent/prompt, data/eval):** Retrieve representative evidence and count repetitions, retaining exceptions. These groups are compression candidates and investigation leads. They do not justify closing every matching alert or labelling rare alerts malicious.

**For the prompt, naming no alert, host or person:** most alerts repeat a few patterns, but repetition doesn't prove an alert is safe and rarity doesn't prove it's malicious. Judge what happened: whether a control blocked it, whether it ran, and what the same user, host or session did afterwards. Look for what differs from the usual pattern: a new source, account, parent process or destination.

## 3. Identities

How do people, machines and IPs appear in each log? Does the mock API's q search ever return the wrong rows, or miss rows?

Affects: data/search-logs

In [5]:
identity_columns = {s:[c for c in d if re.search(
    r"user|host|ip$|IPAddress|Identity|recipient|sender|customer|session|actor|subject|target|runner|^dc$", c)]
    for s,d in logs.items()}
show("Identity-bearing fields by source (nested fields still need parsing)", pd.DataFrame(
    [[s, ", ".join(c)] for s,c in identity_columns.items()], columns=["source","fields"]))
prefix_pairs = [(a,b) for a in users.username for b in users.username if a != b and b.startswith(a)]
show("Directory collisions", {"username_prefix_pairs":len(prefix_pairs),
    "nonunique_display_names": int(users.display_name.duplicated(keep=False).sum()),
    "duplicate_upns":int(users.upn.str.casefold().duplicated().sum()),
    "duplicate_asset_ips":int(assets.ip.duplicated().sum())})
# Reproduce q exactly as implemented, without its limit, then compare to exact aliases.
query = "laura.ferrer"
u = users[users.username.eq(query)].iloc[0]
d = logs["idp_signins"]
q_mask = pd.Series([query in encoded(r).lower() for r in raw_records(d)], index=d.index)
e_mask = exact_mask(d, [u.username, u.upn])
show("Username substring vs exact directory aliases", pd.DataFrame([
    [query, int(q_mask.sum()), int(e_mask.sum()), int((q_mask & ~e_mask).sum()), int((e_mask & ~q_mask).sum())]
], columns=["q","substring_rows","exact_rows","extra_rows","missed_rows"]))
# A machine name is absent in firewall records, which identify it by IP.
asset = assets[assets.hostname.eq("JUMP01")].iloc[0]
f = logs["firewall"]
q_host = pd.Series([asset.hostname.lower() in encoded(r).lower() for r in raw_records(f)], index=f.index)
ip_match = f.src_ip.eq(asset.ip) | f.dst_ip.eq(asset.ip)
show("Hostname lookup against IP-only firewall", {"hostname":"JUMP01", "q_rows":int(q_host.sum()),
     "exact_ip_rows":int(ip_match.sum()), "missed_by_hostname":int((ip_match & ~q_host).sum())})
gaps = [[s, c, int(logs[s][c].fillna("").eq("").sum()), len(logs[s])] for s, cols in identity_columns.items() for c in cols
        if logs[s][c].map(lambda x: isinstance(x, str) or x is None or x != x).all()]
show("Fields with empty values (session_id and assigned_ip are not identities)", pd.DataFrame(
    [row for row in gaps if row[2]], columns=["source", "field", "empty", "rows"]))

Identity-bearing fields by source (nested fields still need parsing)
      source                                     fields
   ad_events              dc, subject, target, src_host
  ai_gateway            session_id, customer_id, src_ip
 cicd_builds                                     runner
  cloudtrail   sourceIPAddress, userAgent, userIdentity
         dlp                             user, hostname
         dns                           src_ip, hostname
ecom_account                        customer_id, src_ip
   ecom_auth            customer_id, src_ip, user_agent
  edr_events                             hostname, user
       email     sender, recipients, subject, sender_ip
email_clicks                               user, src_ip
    firewall                             src_ip, dst_ip
   idp_audit                  actor_upn, target, src_ip
 idp_signins   user_upn, src_ip, user_agent, session_id
       proxy src_ip, hostname, user, dst_ip, user_agent
   sql_audit                       

Directory collisions


{'username_prefix_pairs': 149, 'nonunique_display_names': 281, 'duplicate_upns': 0, 'duplicate_asset_ips': 0}



Username substring vs exact directory aliases
           q  substring_rows  exact_rows  extra_rows  missed_rows
laura.ferrer             153          73          80            0

Hostname lookup against IP-only firewall
{'hostname': 'JUMP01', 'q_rows': 0, 'exact_ip_rows': 220, 'missed_by_hostname': 220}



Fields with empty values (session_id and assigned_ip are not identities)
     source       field  empty   rows
 edr_events        user      7 112492
idp_signins  user_agent      1  76713
idp_signins  session_id   3800  76713
        vpn assigned_ip    167   9821



In [6]:
def identity_shape(value):
    if not isinstance(value, str) or not value:
        return "empty"
    if "\\" in value:
        return "DOMAIN\\user"
    if "@" in value:
        return "UPN or email"
    if re.fullmatch(r"\d+\.\d+\.\d+\.\d+", value):
        return "IP"
    if re.fullmatch(r"[A-Z0-9-]+", value):
        return "uppercase name"
    if re.fullmatch(r"[a-z0-9_.-]+", value):
        return "bare username"
    return "other"
named_fields = {"idp_signins": ["user_upn"], "idp_audit": ["actor_upn"], "email": ["sender"], "email_clicks": ["user"],
                "vpn": ["user"], "proxy": ["user", "hostname"], "edr_events": ["user", "hostname"],
                "dlp": ["user", "hostname"], "sql_audit": ["user", "client_host"], "ad_events": ["subject", "src_host"],
                "dns": ["hostname"]}
format_rows = []
for source, columns in named_fields.items():
    for column in columns:
        shapes = logs[source][column].map(identity_shape).value_counts()
        format_rows.append([source, column, shapes.index[0], int(shapes.iloc[0]), len(logs[source]) - int(shapes.iloc[0])])
show("How people and machines appear in each log", pd.DataFrame(
    format_rows, columns=["source", "field", "main_format", "rows", "other_rows"]))
show("Logs without a plain name field", {"firewall": "IPs only",
     "cloudtrail": "userIdentity object with " + ", ".join(logs["cloudtrail"].userIdentity.iloc[0]),
     "email": "recipients is a list of UPNs"})
# One person in every log, searched the way the mock's q does it.
person = users[users.username.eq("laura.ferrer2")].iloc[0]
search_rows = []
for source, d in logs.items():
    text = [encoded(r).lower() for r in raw_records(d)]
    by_upn, by_username = sum(person.upn.lower() in t for t in text), sum(person.username.lower() in t for t in text)
    if by_upn or by_username:
        search_rows.append([source, by_upn, by_username, by_username - by_upn])
person_search = pd.DataFrame(search_rows, columns=["source", "q_upn", "q_username", "missed_by_upn"])
show("One person, searched by UPN and by username", person_search)
show("Rows missed by searching only the UPN", int(person_search.missed_by_upn.sum()))
show("Accounts sharing this display name", users[users.display_name.eq(person.display_name)][["username", "department"]])

How people and machines appear in each log
      source       field    main_format   rows  other_rows
 idp_signins    user_upn   UPN or email  76713           0
   idp_audit   actor_upn   UPN or email    762           0
       email      sender   UPN or email  56797           0
email_clicks        user   UPN or email  16249           0
         vpn        user  bare username   9821           0
       proxy        user  bare username 194122         136
       proxy    hostname uppercase name 194258           0
  edr_events        user    DOMAIN\user 112482          10
  edr_events    hostname uppercase name 112492           0
         dlp        user  bare username   1042           0
         dlp    hostname uppercase name   1042           0
   sql_audit        user  bare username   3924           0
   sql_audit client_host uppercase name   3924           0
   ad_events     subject  bare username    526           0
   ad_events    src_host uppercase name    526           0
         dns 

One person, searched by UPN and by username
      source  q_upn  q_username  missed_by_upn
   ad_events      0           1              1
  edr_events      0          95             95
       email     57          57              0
email_clicks     15          15              0
   idp_audit     43          43              0
 idp_signins     80          80              0
       proxy      0         179            179
         vpn      0           6              6

Rows missed by searching only the UPN
281

Accounts sharing this display name
     username department
 laura.ferrer   Finanzas
laura.ferrer2 E-commerce



**Observed:** The concrete username query can mix separate identities; hostname search cannot find IP-only firewall rows. Neither result is an API outage. People appear as a UPN in some logs, a bare username in others and `DOMAIN\user` in EDR, so a search on one form misses the rest: the table above shows the count for one person. Two accounts share the display name of the traced case, in different departments. `userIdentity`, `details`, recipient arrays and customer/session identifiers require source-specific extraction.

**Design interpretation (data/search-logs):** Resolve exact UPN/username/hostname/IP aliases per source before searching. Filter returned rows by parsed field equality; do not use a display name as a join key. Searching a hostname alone is insufficient. Substring counts here precede the API's limit; section 12 checks the actual route and shows that q can't match `DOMAIN\user` typed normally.

## 4. Links

Is event_id unique across logs? Do related_events always resolve? Are logs sorted by time, and what does the API's limit cut off?

Affects: data/search-logs, agent/triage

In [7]:
event_index = collections.defaultdict(list)
link_stats = []
for source, d in logs.items():
    for i, event_id in d.event_id.items():
        event_index[event_id].append((source, i))
    link_stats.append([source,len(d),int(d.event_id.duplicated().sum()),
        int(d.timestamp_utc.isna().sum()), int(d.timestamp_utc.diff().lt(pd.Timedelta(0)).sum())])
refs = [(a.alert_id, event_id) for a in alerts.itertuples() for event_id in a.related_events]
missing_refs = [(aid,eid) for aid,eid in refs if eid not in event_index]
ambiguous_refs = [(aid,eid) for aid,eid in refs if len(event_index.get(eid, [])) > 1]
show("Event identity and chronological checks", pd.DataFrame(link_stats,
    columns=["source","rows","duplicate_ids_within","invalid_time","backward_steps"]))
show("Link resolution", {"reference_occurrences":len(refs), "unique_referenced_ids":len(set(e for _,e in refs)),
    "cross_source_or_within_duplicate_ids":sum(len(v)>1 for v in event_index.values()),
    "missing_reference_occurrences":len(missing_refs), "ambiguous_reference_occurrences":len(ambiguous_refs),
    "unreferenced_log_events":sum(map(len,logs.values()))-len({e for _,e in refs if e in event_index})})
if missing_refs:
    show("Missing reference examples (IDs only)",pd.DataFrame(missing_refs[:8],columns=["alert_id","event_id"]))
center = pd.Timestamp("2026-10-07T08:35:15Z")
cutoff_rows=[]
for source in ["proxy","idp_signins","idp_audit","email"]:
    d=logs[source]
    part=d[d.timestamp_utc.between(center-pd.Timedelta(hours=24),center+pd.Timedelta(hours=24))]
    cutoff_rows.append([source,len(part),min(200,len(part)),max(0,len(part)-200),
        part.timestamp_utc.iloc[199].isoformat() if len(part)>=200 else "not reached"])
show("API first-200 cutoff in one unfiltered ±24 h window",pd.DataFrame(cutoff_rows,
    columns=["source","matching_rows","returned","omitted","last_returned_utc"]))
show("Alert ingestion lag vs latest referenced event (minutes)",pd.Series([
    (a.created_at_utc-max(logs[s].loc[i,"timestamp_utc"] for e in a.related_events for s,i in event_index.get(e,[]))).total_seconds()/60
    for a in alerts.itertuples() if any(e in event_index for e in a.related_events)
]).describe(percentiles=[.5,.95]).round(2).to_frame("minutes").reset_index(names="stat"))

lag_rows=[]
for a in alerts.itertuples():
    times=[logs[s].loc[i,"timestamp_utc"] for e in a.related_events for s,i in event_index.get(e,[])]
    if times and max(times)>a.created_at_utc:
        lag_rows.append([a.alert_id,a.rule_id,round((max(times)-a.created_at_utc).total_seconds()/60,2)])
show("Alerts linking events later than alert creation",{"alerts":len(lag_rows),
    "alerts_without_related_events":int(alerts.related_events.map(len).eq(0).sum())})
future_links=pd.DataFrame(lag_rows,columns=["alert_id","rule_id","minutes_after_creation"])
show("Later-event links by rule (do not silently rewrite timestamps)",future_links.groupby("rule_id").minutes_after_creation.agg(["size","median","max"]).round(1).reset_index())
show("Alert with no linked events",alerts[alerts.related_events.map(len).eq(0)][["alert_id","rule_id","severity"]])
linked_times=[[logs[s].loc[i,"timestamp_utc"] for e in a.related_events for s,i in event_index.get(e,[])] for a in alerts.itertuples()]
event_start=pd.to_datetime(pd.Series([min(t) if t else pd.NaT for t in linked_times],index=alerts.index),utc=True)
event_end=pd.to_datetime(pd.Series([max(t) if t else pd.NaT for t in linked_times],index=alerts.index),utc=True)

Event identity and chronological checks
      source   rows  duplicate_ids_within  invalid_time  backward_steps
   ad_events    526                     0             0               0
  ai_gateway   8817                     0             0               0
 cicd_builds    559                     0             0               0
  cloudtrail   6496                     0             0               0
         dlp   1042                     0             0               0
         dns 178221                     0             0               0
ecom_account   7661                     0             0               0
   ecom_auth  44064                     0             0               0
  edr_events 112492                     0             0               0
       email  56797                     0             0               0
email_clicks  16249                     0             0               0
    firewall  24648                     0             0               0
   idp_audit    762     

Alerts linking events later than alert creation
{'alerts': 336, 'alerts_without_related_events': 1}

Later-event links by rule (do not silently rewrite timestamps)
   rule_id  size  median   max
UC-EDR-005     1     2.0   2.0
UC-IDP-002     1     5.1   5.1
UC-NET-001   334     9.1 174.6

Alert with no linked events
           alert_id    rule_id severity
ALR-20261021-011900 UC-NET-001   medium



**Observed:** The tables check every event ID and every explicit alert reference. File order is measured rather than assumed: logs are oldest first. `related_events` is a selected subset of evidence, not a complete incident membership list. In the displayed windows, taking the first 200 omits later evidence. Almost all alerts that link later events come from one rule, and one alert links nothing.

**Design interpretation (data/search-logs, agent/triage):** Store source plus event ID as provenance. Check `truncated`, split saturated time windows and deduplicate boundary results; the log endpoint has no offset. Correlate using event timestamps, preserving alert creation time separately, and don't assume linked events came before the alert.

**For truncated results (agent/triage):** the rows a limit removes are the later ones, so a truncated search can't show that nothing happened afterwards. It must not support the policy's −1, and the ticket must say the evidence is incomplete. The same applies to rows cut to fit a prompt. It can still support a +1.

## 5. Context size

How many rows and KB do the logs from 24 hours either side give, per source? Which sources matter for each rule or runbook? How much must be trimmed for a prompt under 8,000 tokens?

Affects: data/context, agent/prompt

In [8]:
# Exact serialized UTF-8 sizes for whole event records as the API would supply.
# Includes field names/JSON punctuation. No tokenizer or model is called.
window_sizes, size_arrays = [], {}
centers=alerts.created_at_utc.sort_values()
for source,d in logs.items():
    ordered=d.sort_values("timestamp_utc")
    times=pd.DatetimeIndex(ordered.timestamp_utc)
    lengths=pd.Series([len(encoded(r).encode("utf-8")) for r in raw_records(ordered)])
    prefix=pd.concat([pd.Series([0]), lengths.cumsum()],ignore_index=True).to_numpy()
    left=times.searchsorted(centers-pd.Timedelta(hours=24),side="left")
    right=times.searchsorted(centers+pd.Timedelta(hours=24),side="right")
    counts=pd.Series(right-left)
    kb=pd.Series((prefix[right]-prefix[left])/1024)
    size_arrays[source]=(counts,kb)
    window_sizes.append([source,int(counts.median()),int(counts.quantile(.95)),round(kb.median(),1),
                         round(kb.quantile(.95),1),round(kb.max(),1)])
show("All-alert ±24 h windows, before entity filtering",pd.DataFrame(window_sizes,
    columns=["source","rows_p50","rows_p95","KB_p50","KB_p95","KB_max"]))
total_rows=sum((v[0] for v in size_arrays.values()))
total_kb=sum((v[1] for v in size_arrays.values()))
show("Combined all-source ±24 h context over all alert centres",{
    "median_rows":int(total_rows.median()),"p95_rows":int(total_rows.quantile(.95)),
    "median_KB":round(total_kb.median(),1),"p95_KB":round(total_kb.quantile(.95),1)})


# Every alert, not one per runbook: exact raw entities, the runbook's sources, +-24 h.
# DATA_BUDGET is measured on agent/core: src/agent/prompts.py allows 28,000 characters,
# and its instructions, policy, runbook and company rules leave at least 14,000 for data.
DATA_BUDGET=14_000
shared=logs["idp_signins"].groupby("src_ip").user_upn.nunique()
shared_ips={ip.casefold() for ip in shared[shared>20].index}
show("IPs used by more than 20 identities in idp_signins",{"ips":len(shared_ips),"most_identities_on_one_ip":int(shared.max()),
     "share_of_sign_in_rows":round(logs["idp_signins"].src_ip.str.casefold().isin(shared_ips).mean(),3)})
entity_sets={a["alert_id"]:{v.casefold() for v in entity_values(a)} for a in alerts.to_dict("records")}
wanted=set().union(*entity_sets.values())
source_index={s:indexed_source(logs[s],wanted) for s in sorted({s for v in RUNBOOK_SOURCES.values() for s in v})}
def alert_context(exclude=frozenset()):
    rows=[]
    for a in alerts.itertuples():
        count,chars=0,0
        for s in RUNBOOK_SOURCES.get(a.runbook,[]):
            n,c=context_for(a.created_at_utc,entity_sets[a.alert_id]-exclude,source_index[s])
            count+=n; chars+=c
        rows.append([a.alert_id,a.rule_id,a.runbook,count,chars])
    return pd.DataFrame(rows,columns=["alert_id","rule_id","runbook","rows","chars"])
with_shared,without_shared=alert_context(),alert_context(shared_ips)
def by_runbook(frame,suffix):
    grouped=frame.groupby("runbook")
    return pd.DataFrame({"kchars_p50"+suffix:(grouped.chars.median()/1000).round(1),
                         "kchars_p95"+suffix:(grouped.chars.quantile(.95)/1000).round(1),
                         "fits"+suffix:grouped.chars.apply(lambda s:round((s<=DATA_BUDGET).mean(),3))})
show("Context per alert against the data budget (raw rows, before any summary)",pd.concat(
    [with_shared.groupby("runbook").size().rename("alerts"),by_runbook(with_shared,""),by_runbook(without_shared,"_no_shared_ips")],axis=1).reset_index())
show("All alerts",{"fits":round((with_shared.chars<=DATA_BUDGET).mean(),3),"fits_no_shared_ips":round((without_shared.chars<=DATA_BUDGET).mean(),3)})
show("Rules most often over budget, without shared IPs",without_shared[without_shared.chars>DATA_BUDGET].rule_id.value_counts().head(8))
identity_example=alerts.set_index("alert_id").loc["ALR-20261018-005343"]
show("Why one identity alert is large: rows per search key",pd.DataFrame(
    [[s,field,context_for(identity_example.created_at_utc,{str(value).casefold()},source_index[s])[0]]
     for field,value in leaves(identity_example.entities) if not field.endswith("display_name")
     for s in RUNBOOK_SOURCES[identity_example.runbook]],columns=["source","searched_field","rows"]).query("rows>0"))

All-alert ±24 h windows, before entity filtering
      source  rows_p50  rows_p95  KB_p50  KB_p95  KB_max
   ad_events        37        46     8.8    10.9    11.5
  ai_gateway       629       639   301.5   305.9   313.8
 cicd_builds        44        72     9.7    15.8    17.3
  cloudtrail       365      1763   153.3   848.6   857.1
         dlp        75        80    22.9    24.6    25.9
         dns     16809     17517  3437.8  3612.2  3653.8
ecom_account       513       658   115.4   148.3   161.3
   ecom_auth      2652      4749   982.3  1817.2  2657.5
  edr_events     10796     10999  5485.0  5593.0  5640.7
       email      5371      5527  2555.1  2640.6  2678.9
email_clicks      1519      1613   416.1   441.4   448.2
    firewall      1906      2281   541.8   644.5   652.7
   idp_audit        62       101    18.0    29.7    31.8
 idp_signins      7298      7486  4309.0  4426.7  4493.1
       proxy     18538     18876  9172.7  9346.8  9761.4
   sql_audit       280       291    63.

Context per alert against the data budget (raw rows, before any summary)
runbook  alerts  kchars_p50  kchars_p95  fits  kchars_p50_no_shared_ips  kchars_p95_no_shared_ips  fits_no_shared_ips
  RB-01    2301        51.2       169.2 0.324                      51.2                     169.2               0.324
  RB-02    2307       535.8       613.0 0.057                     535.8                     613.0               0.057
  RB-03     981      2174.6      2263.4 0.076                       4.8                       6.7               0.995
  RB-04      62       208.1       241.3 0.048                     208.1                     241.3               0.048
  RB-05    3495         0.3         0.4 0.990                       0.3                       0.4               0.990
  RB-06      53        43.3        45.9 0.000                      43.3                      45.9               0.000
  RB-07     416         0.5         0.5 1.000                       0.5                       0.5    

**Observed:** The first table covers every alert centre without an entity filter, so it's an upper limit nobody would send. The later tables cover every alert with exact raw entities; they aren't an alias-complete retrieval benchmark. A few public IPs are shared by many identities: in the identity example almost all rows come from one such IP, not from the user. Without them, identity alerts fit the budget; malware, email, network, data-loss, cloud and build alerts mostly still don't. Sources are mapped from runbook checks; rule → runbook comes from each alert and the catalogue.

**Design interpretation (data/context, agent/prompt):** The 28,000-character limit leaves about 14,000 for data. Don't search logs on IPs shared by many identities; report their row count instead. Whether they're office connections isn't established here.

**Order and trimming (agent/prompt):** the model reads the alert first, then what code already knows (criticality, users, assets, changes), then raw rows. Rows are cut from the end of each list and logs are oldest first, so unranked lists lose the later events, which the −1 rule and the traced case depend on. Data must send rows most important first: linked events, then changes of state, then repeats as counts. Token counts remain estimates, and actual model tokenisation and answer quality remain unverified.

## 6. Severity

How many alerts name a known machine, only a user, or nothing internal? Apply the policy matrix. Where do the policy, runbooks, deck and sample ticket disagree?

Affects: data/severity

In [9]:
severity_order=["informational","low","medium","high","critical"]
criticality_order=["Baja","Media","Alta","Crítica"]
matrix=pd.DataFrame([
 ["informational","informational","low","low"],
 ["informational","low","low","medium"],
 ["low","medium","medium","high"],
 ["medium","high","high","critical"],
 ["high","critical","critical","critical"]],index=severity_order,columns=criticality_order)
show("Policy matrix (before bounded adjustment)",matrix.reset_index(names="rule_severity"))
asset_alias={str(a[k]).casefold():a for a in assets.to_dict("records") for k in ["hostname","ip"] if a[k]}
user_alias={str(u[k]).casefold():u for u in users.to_dict("records") for k in ["upn","username"] if u[k]}
# Order in docs/decisions.md: machine named, user's workstation, the service the rule targets, Media.
# RB-05 is "Tienda online y app" in servicios_negocio.csv, and the supplied example ticket also treats it as critical.
SERVICE_CRITICALITY={"RB-05":"Crítica"}
def highest(rows):
    return max((x["criticality"] for x in rows),key=criticality_order.index,default=None)
severity_rows,order_inputs=[],[]
for a in alerts.to_dict("records"):
    vals={v.casefold() for v in entity_values(a)}
    named_assets={asset_alias[v]["hostname"]:asset_alias[v] for v in vals if v in asset_alias}
    named_users={user_alias[v]["upn"]:user_alias[v] for v in vals if v in user_alias}
    workstation_assets={u["workstation"]:asset_alias[u["workstation"].casefold()] for u in named_users.values()
                        if u["workstation"].casefold() in asset_alias}
    category="known_machine" if named_assets else "only_known_user" if named_users else "no_resolved_internal_entity"
    asset,workstation=highest(named_assets.values()),highest(workstation_assets.values())
    service=SERVICE_CRITICALITY.get(a["runbook"])
    crit,source=(asset,"asset") if asset else (workstation,"workstation") if workstation else (service,"service") if service else ("Media","default")
    vip=any(u["vip"]=="True" or u["privileged"]=="True" for u in named_users.values())
    base=matrix.loc[a["severity"],crit]
    severity_rows.append([a["alert_id"],a["rule_id"],category,crit,source,vip,a["severity"],base,a["evidence"].get("action")=="blocked"])
    order_inputs.append([a["severity"],a["rule_id"],a["runbook"],asset,workstation])
severity_calc=pd.DataFrame(severity_rows,columns=["alert_id","rule_id","entity_resolution","criticality","criticality_source",
    "vip_or_privileged","rule_severity","matrix_base","control_blocked"])
changed=severity_calc.matrix_base.ne(severity_calc.rule_severity)
show("Entity resolution coverage",severity_calc.entity_resolution.value_counts())
show("Criticality source",severity_calc.criticality_source.value_counts())
show("Media and Alta give different results for these rule severities present in the alerts",
     [s for s in alerts.severity.unique() if matrix.loc[s,"Media"]!=matrix.loc[s,"Alta"]])
show("Matrix base vs rule severity",pd.crosstab(severity_calc.rule_severity,severity_calc.matrix_base).reset_index())
blocked_changes=changed&severity_calc.rule_id.eq("UC-WEB-001")&severity_calc.control_blocked
show("How much the matrix really moves",{"changed_by_matrix":int(changed.sum()),
     "changed_by_source":severity_calc[changed].criticality_source.value_counts().to_dict(),
     "blocked_web_alerts_a_minus_1_returns_to_rule_severity":int(blocked_changes.sum()),
     "still_changed_after_that":int((changed&~blocked_changes).sum()),
     "vip_or_privileged_alerts":int(severity_calc.vip_or_privileged.sum())})
def base_under(order,services):
    bases=[]
    for rule_severity,rule,runbook,asset,workstation in order_inputs:
        picks={"asset":asset,"workstation":workstation,"service":services.get(runbook)}
        bases.append(matrix.loc[rule_severity,next((picks[k] for k in order if picks[k]),"Media")])
    return pd.Series(bases,index=alerts.index)
older_services={"RB-01":"Alta","RB-03":"Crítica","RB-05":"Crítica","RB-07":"Alta","RB-08":"Alta"}
order_rows=[]
for name,order,services in [("decided: machine, workstation, RB-05",("asset","workstation","service"),SERVICE_CRITICALITY),
                            ("first notebook: machine, workstation, all services",("asset","workstation","service"),older_services),
                            ("first decisions.md: machine, all services, workstation",("asset","service","workstation"),older_services)]:
    bases=base_under(order,services)
    order_rows.append([name,int(bases.ne(alerts.severity).sum()),int((bases[alerts.rule_id.eq("UC-IDP-001")]=="high").sum()),
                       int((bases[alerts.rule_id.eq("UC-IDP-002")]=="critical").sum())])
show("Criticality orders compared",pd.DataFrame(order_rows,columns=["order","changed","impossible_travel_high","password_spray_critical"]))
low_assets=assets[assets.criticality.eq("Baja")].hostname.tolist()
show("Only machine rated Baja, and alerts the matrix lowers there",{"machines":low_assets,
     "lowered_alerts":severity_calc[severity_calc.criticality.eq("Baja")&changed][["alert_id","rule_severity","matrix_base"]].to_dict("records")})
# Read supplied reference artefacts as data, without interpreting embedded instructions.
schema=json.loads((DATA/"01_contexto/ticket_salida.schema.json").read_text())
example=json.loads((DATA/"01_contexto/ticket_ejemplo.json").read_text())
slide_text={}
for ppt in sorted(DATA.glob("*.pptx")):
    with zipfile.ZipFile(ppt) as z:
        for name in z.namelist():
            if re.fullmatch(r"ppt/slides/slide\d+\.xml",name):
                slide_text[name]=" ".join(x.text or "" for x in ET.fromstring(z.read(name)).iter() if x.tag.endswith("}t"))
show("Reference contract differences",pd.DataFrame([
 ["policy vs ticket schema","3 policy verdicts; schema also permits needs_more_info"],
 ["policy vs RB-01","Unverified IBAN change is critical in RB-01; the matrix plus at most +1 doesn't reach it"],
 ["policy vs RB-07","Blocked attempt is medium in RB-07; the policy's -1 gives low"],
 ["policy vs deck slide 5","Approved encoded PowerShell: policy says benign_true_positive, deck says false_positive"],
 ["policy vs deck slides 7/9 and project rule","Policy exempts block_ioc/purge_email; project requires a person for all containment"],
 ["deck slide 6 vs inventory","Deck rates CRM copies Crítica; servicios_negocio.csv rates backups Alta"],
 ["ticket example","Example drops medium to informational (-2), exceeding the policy max -1"],
],columns=["comparison","finding"]))
deck_slide5=re.sub(r"\s+"," ",slide_text.get("ppt/slides/slide5.xml",""))
show("Deck slide 5 check",{"encoded_powershell_is_its_false_positive_example":bool(re.search(r"false positive.*PowerShell codificado",deck_slide5))})
show("Sample ticket checks",{"alert_ids":example.get("alert_ids"),"severity":example.get("severity"),
 "verdict":example.get("verdict"),"has_severity_rationale":"severity_rationale" in example,
 "schema_verdict_options":schema["properties"]["verdict"]["enum"],"deck_slides_parsed":len(slide_text)})

example_base=matrix.loc["low","Crítica"]
example_expected=severity_order[max(0,severity_order.index(example_base)-1)]
show("Worked ticket arithmetic check",{"matrix_low_x_critical":example_base,
 "after_one_level_block_adjustment":example_expected,"example_severity":example["severity"],
 "example_alert_id_exists":all(aid in set(alerts.alert_id) for aid in example["alert_ids"])})

Policy matrix (before bounded adjustment)
rule_severity          Baja         Media     Alta  Crítica
informational informational informational      low      low
          low informational           low      low   medium
       medium           low        medium   medium     high
         high        medium          high     high critical
     critical          high      critical critical critical

Entity resolution coverage
                      value  count
no_resolved_internal_entity   4026
              known_machine   3818
            only_known_user   3217

Criticality source
      value  count
      asset   3818
    service   3495
workstation   3206
    default    542

Media and Alta give different results for these rule severities present in the alerts
[]

Matrix base vs rule severity
rule_severity  critical  high  low  medium
     critical        12     1    0       0
         high        89   685    0       1
          low         0     0 2866    3406
       medium         0

Criticality orders compared
                                                 order  changed  impossible_travel_high  password_spray_critical
                  decided: machine, workstation, RB-05     3497                       0                        0
    first notebook: machine, workstation, all services     3570                       0                       62
first decisions.md: machine, all services, workstation     4476                     901                       62

Only machine rated Baja, and alerts the matrix lowers there
{'machines': ['PRINT01'], 'lowered_alerts': [{'alert_id': 'ALR-20261018-011895', 'rule_severity': 'high', 'matrix_base': 'medium'}, {'alert_id': 'ALR-20261018-011896', 'rule_severity': 'critical', 'matrix_base': 'high'}]}

Reference contract differences
                                comparison                                                                                  finding
                   policy vs ticket schema                               

**Observed:** The matrix is applied to all alerts under the displayed order. The table is a **base-severity calculation**, not final triage. Users have no criticality field. `Media` and `Alta` give the same result for every alert here, so only `Crítica` and `Baja` change anything. Almost every change comes from treating web alerts as `Crítica`, and blocked web alerts return to their rule severity after the policy's −1. VIP/privilege is a possible +1 adjustment, never multiple increases. Unknown external entities are not automatically low risk. The only `Baja` machine is `PRINT01`, where the matrix lowers two alerts.

**Design interpretation (data/severity):** Code the matrix, provenance and a single bounded adjustment. Use the decided order. An identity's service isn't treated as `Crítica` for every sign-in: the first decisions.md order would raise every impossible-travel alert to `high`. Confirm execution and subsequent activity before raising/lowering severity; apply the authorised-activity override only with validated scope and time. Retain the rule, matrix base, adjustment and final result separately. A rule's critical rating does not establish malice.

**Runbook minimums (agent/triage):** only RB-01's IBAN case states a severity for a rule, so it's the only floor, and it beats the one-level cap. Other runbook severities describe outcomes. RB-07's `medium` for a blocked attempt isn't applied: the policy's −1 stands.

The supplied example assigns `informational` after describing a `medium` matrix base and a −1 adjustment: one step would be `low`. Its alert ID is a placeholder absent from the dataset. Treat this as a concrete reference inconsistency, not an instruction to copy the result. Where the deck and the policy disagree on a verdict, the policy is followed.

## 7. Time

What time zone are alerts, logs and the change calendar in? Check against real alerts that match approved changes, including across the clock change.

Affects: data/context

In [10]:
calendar = change_times(changes)
show("Timestamp formats",pd.DataFrame([
 ["alerts",len(alerts),int(alerts.created_at.astype(str).str.endswith("Z").sum()),int(alerts.created_at_utc.isna().sum())],
 *[[s,len(d),int(d.timestamp.astype(str).str.endswith("Z").sum()),int(d.timestamp_utc.isna().sum())] for s,d in logs.items()],
 ["change calendar start/end",2*len(changes),int(changes[["inicio","fin"]].stack().str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$").sum()),
  int(calendar[["inicio_utc","fin_utc"]].isna().sum().sum())]],columns=["source","values","explicit_offset","invalid_times"]))
# Calendar has no offset: test Madrid local against a fixed UTC+2 interpretation.
scan=alerts[alerts.rule_id.eq("UC-NET-003") & alerts.entities.map(lambda e:
    any(v=="NESSUS01" for _,v in leaves(e)))].copy()
clock_rows=[]
for date in ["2026-10-07","2026-10-14","2026-10-21","2026-10-28"]:
    local=scan.created_at_utc.dt.tz_convert("Europe/Madrid")
    part=scan[local.dt.strftime("%Y-%m-%d").eq(date)]
    cid=calendar.loc[calendar.inicio.str.startswith(date),"id"].iloc[0]
    c=calendar.set_index("id").loc[cid]
    fixed_start=pd.Timestamp(c.inicio).tz_localize("UTC")-pd.Timedelta(hours=2)
    fixed_end=pd.Timestamp(c.fin).tz_localize("UTC")-pd.Timedelta(hours=2)
    clock_rows.append([cid,len(part),part.created_at_utc.min().isoformat(),
       part.created_at_utc.dt.tz_convert("Europe/Madrid").min().isoformat(),
       int(part.created_at_utc.between(c.inicio_utc,c.fin_utc).sum()),
       int(part.created_at_utc.between(fixed_start,fixed_end).sum())])
show("Scanner alerts matching real change windows, before/after DST",pd.DataFrame(clock_rows,
    columns=["change_id","scanner_alerts","first_UTC","first_Madrid","inside_Madrid_window","inside_fixed_plus2_window"]))
nessus_ip=assets.set_index("hostname").ip["NESSUS01"]
scan_rows=logs["firewall"][logs["firewall"].src_ip.eq(nessus_ip)&logs["firewall"].zone_dst.eq("ANY")]
scan_days=scan_rows.groupby(scan_rows.timestamp_utc.dt.tz_convert("Europe/Madrid").dt.date).timestamp_utc
show("Raw scanner traffic by Madrid date",pd.DataFrame({"rows":scan_days.size(),"first_UTC":scan_days.min().astype(str).str[:19],
     "last_UTC":scan_days.max().astype(str).str[:19]}).reset_index(names="Madrid_date"))
show("Madrid clock change",pd.DataFrame([
 [t,t.tz_convert("Europe/Madrid").isoformat()] for t in pd.to_datetime(
     ["2026-10-25T00:30:00Z","2026-10-25T01:30:00Z"],utc=True)
],columns=["UTC","Madrid_local"]))
# Independent approved maintenance example.
maint=alerts[alerts.rule_id.eq("UC-EDR-011") & alerts.evidence.map(lambda e:
    "CHG-2026-1080" in approval_refs(e))]
show("Pre-change maintenance cross-check",{"change":"CHG-2026-1080","alerts":len(maint),
 "inside_Madrid_window":int(change_window(maint.created_at_utc,"CHG-2026-1080",calendar).sum())})

Timestamp formats
                   source  values  explicit_offset  invalid_times
                   alerts   11061            11061              0
                ad_events     526              526              0
               ai_gateway    8817             8817              0
              cicd_builds     559              559              0
               cloudtrail    6496             6496              0
                      dlp    1042             1042              0
                      dns  178221           178221              0
             ecom_account    7661             7661              0
                ecom_auth   44064            44064              0
               edr_events  112492           112492              0
                    email   56797            56797              0
             email_clicks   16249            16249              0
                 firewall   24648            24648              0
                idp_audit     762              762        

**Observed:** All event timestamps checked above have explicit UTC (`Z`). Calendar start/end values have no offset. Raw scanner traffic starts at the same minute each week and moves by exactly one UTC hour after the clock change, matching the Madrid wall-time schedule; this supports interpreting the calendar as `Europe/Madrid`. The README's fixed UTC+2 statement does not hold throughout this period.

**Design interpretation (data/context, agent/prompt):** Preserve UTC for comparisons and convert Madrid calendar wall times using zoneinfo, not a fixed offset, in code rather than in the prompt. Raise on ambiguous or nonexistent local calendar times. Compare against event times. The prompt only needs to say that times are UTC and that code has checked the windows. The examples support this interpretation, but no formal calendar timezone declaration was supplied. The displayed events also fall within the wider fixed-offset windows, so window inclusion alone cannot distinguish those hypotheses; the moved start time supplies the additional support.

## 8. Authorised activity

Which alerts match approved changes, approved flows, HR events or company rules?

Affects: agent/prompt, data/context

In [11]:
# These are reproducible contextual matches, not automatically assigned verdicts.
# Windows use the linked events' time, because an alert can be created after a window closes.
context_matches=[]
authorised_ids=set()
context_ids={}
def context_match(name, mask, constraint, validated=False):
    ids=set(alerts.loc[mask,"alert_id"])
    context_ids[name]=ids
    context_matches.append([name,len(ids),constraint,"scoped match" if validated else "lead"])
    if validated: authorised_ids.update(ids)
    return ids
def event_window(change):
    return change_window(event_start,change,calendar) & change_window(event_end,change,calendar)
has_value=lambda v: alerts.entities.map(lambda e:any(x==v for _,x in leaves(e)))
scanner_mask=alerts.alert_id.isin(scan.alert_id)
scanner_window=pd.Series(False,index=alerts.index)
for cid in ["CHG-2026-1049","CHG-2026-1050","CHG-2026-1051","CHG-2026-1052"]:
    scanner_window |= event_window(cid)
# zone_dst is ANY on scanner rows, so destinations are tested against the OT range itself.
ot_networks=[ipaddress.ip_network(c) for c in inventory["zonas_red"].loc[lambda z:z.zona.eq("LOGISTICS-OT"),"cidr"]]
def in_ot(ip):
    try:
        return any(ipaddress.ip_address(ip) in network for network in ot_networks)
    except ValueError:
        return False
def no_ot_refs(a):
    rows=[logs[s].loc[i] for eid in a.related_events for s,i in event_index.get(eid,[]) if s=="firewall"]
    return bool(rows) and not any(in_ot(r.dst_ip) for r in rows)
scan_scoped=alerts.apply(no_ot_refs,axis=1)
context_match("authorised scanner",scanner_mask & scanner_window & scan_scoped,
              "NESSUS01 + Wednesday change window + destinations outside the OT range",True)
sccm=alerts.rule_id.eq("UC-EDR-002") & alerts.evidence.map(lambda e:
    "CcmExec" in str(e.get("parent","")) and e.get("user")=="SYSTEM")
event_hour=event_start.dt.tz_convert("Europe/Madrid").dt.hour
context_match("SCCM inventory",sccm & event_window("CHG-2026-1031") & event_hour.between(8,9),
              "approved period + stated local 08:00-10:00 by event time + parent + one command; account differs from calendar")
context_match("SCCM pattern outside stated daily window",sccm & event_window("CHG-2026-1031") & ~event_hour.between(8,9),
              "same pattern; verify schedule rather than broadly allowlisting")
pentest_ips={"192.0.2.10","192.0.2.11","192.0.2.12"}
pentest=alerts.rule_id.eq("UC-WEB-001") & alerts.entities.map(lambda e:bool(set(e.get("ips",[])) & pentest_ips))
context_match("pentest",pentest & event_window("CHG-2026-1042"),
              "approved IP + window + host; 'no DoS' isn't checked")
ps=alerts.rule_id.eq("UC-NET-004") & has_value("JUMP01") & alerts.evidence.map(lambda e:
    e.get("dst") in {"ERP-APP01","HR-APP01","PRINT01","CRM-APP01"})
context_match("PsExec patching",ps & event_window("CHG-2026-1057"),
              "JUMP01 + listed target + approved window; identity/method needs review")
flow=alerts.rule_id.eq("UC-DLP-002") & alerts.evidence.map(lambda e:e.get("recipient")=="asesoria@gestoria-pons.example")
context_match("payroll flow",flow & event_window("FLUJO-0007"),
              "recipient + active flow; verify payroll sender/contents")
context_match("OT allowed company source",alerts.rule_id.eq("UC-OT-001") & (has_value("ENG-WS01")|has_value("WMS01")),
              "company authorised source; never execute OT containment",True)
context_match("OT maintenance",alerts.rule_id.eq("UC-OT-001") & has_value("JUMP01") & event_window("CHG-2026-1063"),
              "source/window; confirm temporary firewall rule and PLC scope")
context_match("directory group change",alerts.rule_id.eq("UC-IDP-008") & has_value("JUMP01") & event_window("CHG-2026-1078"),
              "source/window/change reference; verify named group/member")
context_match("EDR agent upgrade",alerts.alert_id.isin(maint.alert_id) & event_window("CHG-2026-1080"),
              "server window + calendar ID; verify short stop and restart")
vpn_corporate=has_value("198.18.240.10")
context_match("WMS supplier exception",alerts.rule_id.eq("UC-VPN-001") & has_value("ext.logisoft@tarvia.example") &
              vpn_corporate & event_window("CHG-2026-1071"),
              "supplier + corporate IP + exceptional window; inspect WMS-only access")
show("Authorised-activity/context candidates",pd.DataFrame(context_matches,
    columns=["context","alerts","constraints_checked_or_remaining","status"]))
# SCCM: alert time against event time, and the calendar's account against the logged one.
sccm_alerts=alerts[sccm]
sccm_rows=logs["edr_events"][logs["edr_events"].event_id.isin({e for r in sccm_alerts.related_events for e in r})]
alert_hour=sccm_alerts.created_at_utc.dt.tz_convert("Europe/Madrid").dt.hour
show("SCCM checks",{"alerts":len(sccm_alerts),"outside_08_10_by_alert_time":int((~alert_hour.between(8,9)).sum()),
     "outside_08_10_by_event_time":int((~event_hour[sccm_alerts.index].between(8,9)).sum()),
     "calendar_account":"svc_sccm","logged_account":sccm_rows.user.value_counts().to_dict(),
     "distinct_commands":int(sccm_rows.command_line.nunique()),"hosts":int(sccm_rows.hostname.nunique()),
     "asset_types":sccm_rows.hostname.map(assets.set_index("hostname").type).value_counts().to_dict()})
scanner_rows=logs["firewall"][logs["firewall"].event_id.isin({e for r in scan.related_events for e in r})]
show("Scanner: destination checks on linked rows",{"rows":len(scanner_rows),"zone_dst":scanner_rows.zone_dst.value_counts().to_dict(),
     "destinations_in_OT_range":int(scanner_rows.dst_ip.map(in_ot).sum())})
pentest_rows=logs["waf"][logs["waf"].src_ip.isin(pentest_ips)]
show("Pentest: raw WAF rows from the approved IPs",{"rows":len(pentest_rows),
     "inside_window":int(change_window(pentest_rows.timestamp_utc,"CHG-2026-1042",calendar).sum()),
     "hosts":pentest_rows.host.value_counts().to_dict(),"action":pentest_rows.action.value_counts().to_dict()})
# Company rules from empresa_tarvia.md, tested against logs.
adm=set(users.username[users.username.str.startswith("adm.")])
edr=logs["edr_events"]
adm_edr=edr[edr.user.astype(str).str.split("\\").str[-1].isin(adm)]
approved_psexec=adm_edr.hostname.isin({"ERP-APP01","HR-APP01","PRINT01","CRM-APP01"}) & change_window(adm_edr.timestamp_utc,"CHG-2026-1057",calendar)
off_edr=adm_edr[adm_edr.hostname.ne("JUMP01") & ~approved_psexec][["hostname","timestamp_utc"]]
ad=logs["ad_events"]
off_ad=ad[ad.subject.isin(adm) & ad.src_host.ne("JUMP01")][["src_host","timestamp_utc"]].rename(columns={"src_host":"hostname"})
off_rule=pd.concat([off_edr,off_ad])
show("adm.* rows outside JUMP01 and outside the PsExec change (lead)",off_rule.groupby("hostname").timestamp_utc.agg(["size","min","max"]).astype(str).reset_index())
backup_alerts=alerts[alerts.entities.map(lambda e:any("svc_backup" in str(v) for _,v in leaves(e)))]
backup_signins=logs["idp_signins"][logs["idp_signins"].user_upn.str.startswith("svc_backup")]
logisoft=logs["vpn"][logs["vpn"].user.eq("ext.logisoft")]
logisoft_local=logisoft.timestamp_utc.dt.tz_convert("Europe/Madrid")
vendor_alerts=alerts.rule_id.eq("UC-VPN-001")
anydesk=edr[edr.process_name.astype(str).str.contains("anydesk",case=False)]
show("Company rules checked against logs (leads)",pd.DataFrame([
 ["adm.* accounts only from JUMP01",f"{len(off_rule)} rows elsewhere, outside the PsExec change"],
 ["svc_backup excluded from MFA",f"{len(backup_signins)} sign-in; alerts: {', '.join(backup_alerts.alert_id)}. An exemption doesn't authorise a login"],
 ["Logisoft Monday-Friday 08:00-18:00",f"{int((~(logisoft_local.dt.day_of_week.lt(5)&logisoft_local.dt.hour.between(8,17))).sum())} of {len(logisoft)} VPN rows outside hours; UC-VPN-001: {int((vendor_alerts&vpn_corporate).sum())} from the corporate IP, {int((vendor_alerts&~vpn_corporate).sum())} from another"],
 ["Only TeamViewer Corporate is approved",f"{len(anydesk)} AnyDesk rows on {anydesk.hostname.nunique()} host; alerts: {', '.join(alerts[alerts.rule_id.eq('UC-EDR-005')].alert_id)}"],
],columns=["company_rule","what_the_logs_show"]))
# HR events with their dates, for the rules whose runbooks ask about them (RB-03 and RB-04).
def hr_matches(upn):
    keys={upn.casefold(),upn.split("@")[0].casefold()}
    return alerts.entities.map(lambda e:any(str(v).casefold() in keys for _,v in leaves(e)))
def stated_period(detail):
    found=re.search(r"del (\d{2})/(\d{2}) al (\d{2})/(\d{2})",detail)
    if not found: return None
    d1,m1,d2,m2=found.groups()
    return pd.Timestamp(f"2026-{m1}-{d1}",tz="Europe/Madrid"),pd.Timestamp(f"2026-{m2}-{d2}",tz="Europe/Madrid")+pd.Timedelta(days=1)
hr_rows=[]
for r in hr.itertuples():
    if r.tipo in {"viaje","teletrabajo_temporal"}:
        period=stated_period(r.detalle)
        for i in alerts.index[alerts.rule_id.eq("UC-IDP-001") & hr_matches(r.usuario)]:
            relation="no end date stated" if period is None else "inside stated dates" if period[0]<=event_start[i]<period[1] else "outside stated dates"
            hr_rows.append([r.tipo,r.usuario.split("@")[0],"UC-IDP-001",alerts.alert_id[i],relation])
    if r.tipo=="baja_voluntaria_comunicada":
        last=pd.Timestamp(r.fecha_efectiva,tz="Europe/Madrid")+pd.Timedelta(days=1)
        for i in alerts.index[alerts.rule_id.str.startswith("UC-DLP") & hr_matches(r.usuario)]:
            hr_rows.append([r.tipo,r.usuario.split("@")[0],alerts.rule_id[i],alerts.alert_id[i],"before last day" if event_start[i]<last else "after last day"])
hr_cases=pd.DataFrame(hr_rows,columns=["HR_type","user","rule","alert_id","relation"])
show("HR events the runbooks ask about, with dates (leads, not verdicts)",hr_cases)

calendar_ids=set(changes.id)
claimed_refs=collections.Counter(ref for e in alerts.evidence for ref in approval_refs(e))
show("Approval references in alert evidence",pd.DataFrame([
 [ref,count,ref in calendar_ids] for ref,count in sorted(claimed_refs.items())],
 columns=["reference","alert_occurrences","exists_in_calendar"]))

# Apply the policy's authorised-activity severity cap only to the two scoped
# analytical matches above. This is a candidate calculation for human review.
scoped_base=severity_calc[severity_calc.alert_id.isin(authorised_ids)].copy()
scoped_base["candidate_authorised_severity"]=scoped_base.matrix_base.map(
    lambda value:"informational" if value=="informational" else "low")
show("Scoped-match policy override candidates",pd.crosstab(
    scoped_base.matrix_base,scoped_base.candidate_authorised_severity).reset_index())

Authorised-activity/context candidates
                                 context  alerts                                                                               constraints_checked_or_remaining       status
                      authorised scanner      19                                         NESSUS01 + Wednesday change window + destinations outside the OT range scoped match
                          SCCM inventory    1349 approved period + stated local 08:00-10:00 by event time + parent + one command; account differs from calendar         lead
SCCM pattern outside stated daily window       0                                                 same pattern; verify schedule rather than broadly allowlisting         lead
                                 pentest      20                                                            approved IP + window + host; 'no DoS' isn't checked         lead
                         PsExec patching       4                                         JUMP01 

Pentest: raw WAF rows from the approved IPs
{'rows': 834, 'inside_window': 834, 'hosts': {'www.tarvia.example': 834}, 'action': {'blocked': 834}}



adm.* rows outside JUMP01 and outside the PsExec change (lead)
hostname size                       min                       max
    FS01    4 2026-10-10 00:40:04+00:00 2026-10-10 01:05:05+00:00
 PRINT01    3 2026-10-10 00:51:06+00:00 2026-10-18 01:13:00+00:00

Company rules checked against logs (leads)
                         company_rule                                                                   what_the_logs_show
      adm.* accounts only from JUMP01                                          7 rows elsewhere, outside the PsExec change
         svc_backup excluded from MFA       1 sign-in; alerts: ALR-20261013-011816. An exemption doesn't authorise a login
   Logisoft Monday-Friday 08:00-18:00 16 of 30 VPN rows outside hours; UC-VPN-001: 4 from the corporate IP, 6 from another
Only TeamViewer Corporate is approved                                2 AnyDesk rows on 1 host; alerts: ALR-20261012-011829

HR events the runbooks ask about, with dates (leads, not verdicts)
            

**Observed:** Counts above distinguish scoped matches from incomplete contextual leads. By alert creation time some SCCM alerts look late; by event time none are. All of them run under a different account from the one the calendar names. Calendar IDs and dates alone do not validate scope. Only one trip states an end date, so the others can't be checked. HR rows show a dated trip covering an impossible-travel alert and a departing employee's data-loss alerts before their last day; these are leads. `adm.*` rows exist outside JUMP01 and outside the approved change.

**Design interpretation (agent/triage, agent/prompt, data/context):** Treat authorisation as a conjunction of identity, source, destination, action and time, checked by code against **event** time, not alert creation time. Keep unmatched constraints in the ticket. Do not close solely because an alert mentions a change, a known vendor, a corporate tool or HR activity. These counts are not ground-truth benign labels.

**What can authorise (agent/triage):** a change or flow that code matched, and a notified trip whose stated dates cover the event (RB-03 says to close those). A trip with no end date doesn't qualify. Other HR events are context and never authorise; RB-04 asks about departures as a reason for concern. Accepted exemptions such as `svc_backup`'s MFA don't authorise a login. An alert from the permitted OT sources is the rule misfiring (the catalogue excludes them), so it's a false positive rather than an authorised change. Follow the policy's verdict for approved changes: `benign_true_positive`, even where the deck's example says otherwise.

**SCCM account (data/context):** the calendar's `svc_sccm` has to be accepted as the logged `NT AUTHORITY\SYSTEM`, but only with the same parent process, the one known command and the daily window. Otherwise a strict identity check rejects every one of these alerts.

## 9. Attacker-written text

Which free-text fields contain content addressed to analysts or automated systems, hidden formatting, or claims of approval? Do claimed approvals exist in the change calendar? What are the limits of this search?

Affects: agent/prompt, agent/triage

In [12]:
# Heuristic screen of every string leaf, including nested structures.
# Categories overlap. A hit is a candidate for review, not a proven injection.
screens={
 "instruction_override":re.compile(r"(?:ignore|disregard).{0,55}(?:previous|prior|instruction|system)|ignora.{0,55}(?:instrucciones|anteriores|sistema)",re.I|re.S),
 "analyst_or_agent_directive":re.compile(r"(?:analyst|analista|automated|automatizado|agente|SOC).{0,100}(?:cerr|close|benign|false.positive|falso.positivo|aprob|instrucc)|(?:do not escalate|no escalar|cierra.{0,25}alerta)",re.I|re.S),
 "approval_reference":re.compile(r"\b(?:SEC-\d+|CHG-\d{4}-\d+|FLUJO-\d+)\b"),
 "hidden_formatting":re.compile(r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]")}
screens["hidden_HTML"]=re.compile(r"<!--|display\s*:\s*none|visibility\s*:\s*hidden|font-size\s*:\s*0|aria-hidden",re.I)
screens["customer_correction_language"]=re.compile(r"(?:ignora|olvida).{0,60}(?:anterior|pedido|direcci)",re.I|re.S)
screens["role_or_policy_override"]=re.compile(r"system\s*:|<system>|instrucciones.{0,25}sistema|prompt.{0,25}sistema|revela.{0,60}(?:intern|c[oó]dig)|bypass",re.I|re.S)
hits=[]
all_tables={"alerts":alerts,**{"log/"+s:d for s,d in logs.items()},
    **{"inventory/"+s:d for s,d in inventory.items()},**{"intel/"+s:d for s,d in intel.items()}}
for table,d in all_tables.items():
    for row in raw_records(d):
        rid=next((row[k] for k in ["alert_id","event_id","asset_id","user_id","id","sha256"] if k in row),"row-without-id")
        for field,value in leaves(row):
            if not isinstance(value,str):continue
            for category,pattern in screens.items():
                if pattern.search(value):
                    hits.append([table,str(rid),field,category])
text_hits=pd.DataFrame(hits,columns=["table","record_id","field","category"]).drop_duplicates()
show("Heuristic field hits by table/category",text_hits.groupby(["table","category"]).size().reset_index(name="field_hits"))
show("Distinct flagged records and fields",{"records":int(text_hits[["table","record_id"]].drop_duplicates().shape[0]),
 "record_field_pairs":int(text_hits[["table","record_id","field"]].drop_duplicates().shape[0])})
# Sample one record per field/category; emit paths/IDs and controlled descriptions only.
examples=text_hits.sort_values(["category","table","record_id"]).groupby(["category","table","field"]).head(1)
show("Sanitised examples (at most 18; never raw content)",examples.head(18))
show("Claimed approval with no calendar record",pd.DataFrame([
 [a.alert_id,"evidence.service_description","SEC-2210","service metadata claims security approval; no calendar entry"]
 for a in alerts.itertuples() if "SEC-2210" in approval_refs(a.evidence)
],columns=["alert_id","field","claimed_reference","description"]))
# Directives outside ai_gateway would be missed by inspecting only Vela prompts.
non_ai=text_hits[~text_hits.table.eq("log/ai_gateway") & text_hits.category.ne("approval_reference")]
show("Non-AI-log heuristic candidates",non_ai.groupby(["table","field","category"]).size().reset_index(name="field_hits"))

show("All screen categories, including zero hits",text_hits.category.value_counts().reindex(
    screens,fill_value=0))
show("Hidden HTML candidate examples (IDs and fields only)",text_hits[
    text_hits.category.eq("hidden_HTML")].groupby(["table","field"]).head(1))
show("Absent claimed approval references (alert evidence)",pd.DataFrame([
    [reference,count] for reference,count in sorted(claimed_refs.items()) if reference not in calendar_ids
],columns=["claimed_reference","alerts"]))


Heuristic field hits by table/category
                       table                     category  field_hits
                      alerts   analyst_or_agent_directive          12
                      alerts           approval_reference          23
                      alerts customer_correction_language         112
                      alerts                  hidden_HTML          12
                      alerts         instruction_override         415
                      alerts      role_or_policy_override         416
inventory/calendario_cambios           approval_reference          13
              log/ai_gateway customer_correction_language         112
              log/ai_gateway      role_or_policy_override           1
              log/edr_events           approval_reference           3
              log/edr_events      role_or_policy_override        1349
                   log/email   analyst_or_agent_directive          45
                   log/email           approval_ref

**Observed:** This lexical scan emits only field paths, record IDs and controlled descriptions. The three TPV alerts' approval reference does not exist among calendar IDs. Repeated representations in alerts and logs are counted separately. Vela attack prompts concern Vela; attempts addressed to a SOC analyst create a different trust-boundary risk.

**Design interpretation (agent/prompt, agent/triage):** Every retrieved text field, including CMDB notes and EDR service descriptions, stays quoted data. Retrieved text cannot approve actions, rewrite policy or supply a verdict. Validate claimed change IDs against calendar scope/time in code. This screen misses paraphrases, encoded payloads, cross-field instructions and unknown languages, and can flag legitimate administrative prose. It is not a complete detector or an estimate of attack prevalence.

The scan also checks HTML comments/CSS hiding and common Spanish customer corrections. These overlap with legitimate language: detector boilerplate in alert descriptions is explicitly not an attack label. Zero Unicode-format hits only means this particular character screen found none. The absent `SEC-4471` claims and hidden email HTML are additional review leads; neither a claim nor visual concealment grants approval. Section 11 lists these alerts for agent tests, which use a fake model; the planted text never goes into prompts as an example.

## 10. Attack stories

Starting from rare, severe alerts and following shared indicators, what are the likely incidents? Fully trace at least one, including evidence the alerts don't link to.

Affects: agent/triage, data/eval

In [13]:
# Start with low-frequency severe rules, then follow exact shared entities/indicators.
rare_rules=set(alerts.rule_id.value_counts()[lambda x:x<=12].index)
seeds=alerts[alerts.rule_id.isin(rare_rules) & alerts.severity.isin(["high","critical"])]
show("Rare severe seed counts (leads, no ground-truth label)",seeds.rule_id.value_counts())
# Exclude ubiquitous services, usernames, process names and private/shared addresses from
# indicator-only incident candidates. Equality is exact, never substring matching.
indicator_paths=("domains[]","files[].sha256","urls[]")
indicator_to_alerts=collections.defaultdict(set)
for a in alerts.itertuples():
    for field,value in leaves(a.entities):
        if field in indicator_paths and value: indicator_to_alerts[(field,str(value))].add(a.alert_id)
shared_groups={tuple(sorted(ids)) for ids in indicator_to_alerts.values() if len(ids)>1 and ids & set(seeds.alert_id)}
show("Exact indicator groups intersecting rare severe seeds",pd.DataFrame([
 [len(ids),len(set(alerts.loc[alerts.alert_id.isin(ids),"rule_id"])),min(ids),max(ids)]
 for ids in sorted(shared_groups,key=lambda ids:(-len(ids),ids))[:10]
],columns=["alerts","rules","first_alert_id","last_alert_id"]))
# Canonical case: trace events rather than relying on alert arrival order.
story_ids=["ALR-20261007-011785","ALR-20261007-011786","ALR-20261007-011787",
           "ALR-20261007-011788","ALR-20261008-011789"]
story=alerts[alerts.alert_id.isin(story_ids)]
story_refs={eid for ids in story.related_events for eid in ids}
account="laura.ferrer2@tarvia.example"
start,end=pd.Timestamp("2026-10-07T00:00Z"),pd.Timestamp("2026-10-09T00:00Z")
# Follow exact user -> session -> audit and exact message_id -> mail/click.
phish=alerts.set_index("alert_id").loc[story_ids[0]]
message_id=phish.evidence["message_id"]
message_rows=logs["email"][logs["email"].message_id.eq(message_id)]
clicks=logs["email_clicks"][logs["email_clicks"].message_id.eq(message_id)]
session_id=alerts.set_index("alert_id").loc[story_ids[2]].evidence["session_id"]
signins=logs["idp_signins"][logs["idp_signins"].session_id.eq(session_id)]
audits=logs["idp_audit"][logs["idp_audit"].actor_upn.eq(account) &
    logs["idp_audit"].timestamp_utc.between(start,end)]
fraud=logs["email"][logs["email"].event_id.isin(
    alerts.set_index("alert_id").loc[story_ids[-1]].related_events)]
transitions=[]
for label,d in [("phishing mail delivered",message_rows),("link click allowed",clicks),
                ("phishing site GET/POST",logs["proxy"][logs["proxy"].event_id.isin(phish.related_events)]),
                ("same session, changed user agent",signins),("mailbox audit",audits),
                ("bank-change message delivered",fraud)]:
    for r in d.itertuples():
        transitions.append([r.timestamp_utc,label,r.event_id,r.event_id in story_refs])
timeline=pd.DataFrame(transitions,columns=["timestamp_UTC","stage","event_id","linked_by_story_alert"])
show("Canonical story: event stage totals",timeline.groupby(["stage","linked_by_story_alert"]).agg(
    events=("event_id","size"),first_UTC=("timestamp_UTC","min"),last_UTC=("timestamp_UTC","max")).reset_index())
show("Canonical story: chronological turning points",timeline.sort_values("timestamp_UTC").groupby("stage").head(2).sort_values("timestamp_UTC"))
show("Same account: audit events after the story window (lead)",logs["idp_audit"][logs["idp_audit"].actor_upn.eq(account) &
    logs["idp_audit"].timestamp_utc.ge(end)][["event_id","operation","timestamp_utc"]].astype(str))
show("Canonical story: audit operations not linked by any story alert",audits[
    ~audits.event_id.isin(story_refs)].operation.value_counts())
show("Canonical story: arrival vs occurrence",pd.DataFrame([
 [a.alert_id,a.rule_id,a.created_at_utc,min(logs[s].loc[i,"timestamp_utc"]
   for e in a.related_events for s,i in event_index.get(e,[]))] for a in story.itertuples()
],columns=["alert_id","rule","alert_created_UTC","earliest_referenced_event_UTC"]))
show("Canonical story: observed scope",{"alerts":len(story),"unique_linked_events":len(story_refs),
 "unlinked_mail_access_events":int((audits.operation.eq("MailItemsAccessed") & ~audits.event_id.isin(story_refs)).sum()),
 "shared_session_distinct_origins":int(signins.src_ip.nunique()),
 "shared_session_distinct_user_agents":int(signins.user_agent.nunique()),
 "fraud_amount_claimed_in_alert":alerts.set_index("alert_id").loc[story_ids[-1]].evidence.get("amount_mentioned"),
 "bank_message_logged":len(fraud)})
# Other chains are explicitly leads. Association is based on exact host or identity;
# it cannot prove the same intruder or that an action succeeded.
lead_specs=[
 ("credential theft to ransomware",["WS-BCN-1770","FS01"],["UC-EDR-001","UC-NET-001","UC-EDR-004","UC-NET-003","UC-NET-004","UC-EDR-007"]),
 ("vendor VPN / DNS / directory / agent tampering",["ext.logisoft@tarvia.example","WMS01","PRINT01"],["UC-VPN-001","UC-NET-006","UC-IDP-008","UC-EDR-011"]),
 ("TPV memory reads and later network",["POS-T07-01","POS-T07-02","POS-T07-03"],["UC-EDR-012","UC-NET-001"]),
 ("possible dependency / cloud chain",["CI-RUNNER01"],["UC-DEV-001"]),
 ("Vela session disclosure",[],["UC-AI-002"]),
]
lead_rows=[]
for name,entities,rules in lead_specs:
    mask=alerts.rule_id.isin(rules)
    if entities:mask &= alerts.entities.map(lambda e:any(v in entities for _,v in leaves(e)))
    part=alerts[mask]
    lead_rows.append([name,len(part),part.created_at_utc.min(),part.created_at_utc.max(),
                      part.alert_id.iloc[0] if len(part) else "none","lead; not fully traced"])
show("Other candidate chains",pd.DataFrame(lead_rows,columns=["hypothesis","alerts","first_UTC","last_UTC","seed_alert_id","status"]))
# Local TIP coverage only. Unknown means absent from this feed, not benign.
feed_values=set(intel["ti_feed"].value)
alert_indicators={v for a in alerts.itertuples() for field,v in leaves(a.entities)
                  if field in {"domains[]","ips[]","files[].sha256"} and isinstance(v,str)}
show("Local intelligence coverage",{"distinct_alert_indicators":len(alert_indicators),
 "present_in_TIP":len(alert_indicators & feed_values),"absent_from_TIP":len(alert_indicators-feed_values)})

show("Raw-log support and limits for the canonical case",pd.DataFrame([
 ["sign-in session",len(signins),"one source IP, two user agents; session changes client, not logged source"],
 ["proxy POST",int((logs["proxy"].event_id.isin(phish.related_events) & logs["proxy"].method.eq("POST")).sum()),
  "submission request logged; payload and actual credential contents unavailable"],
 ["mailbox access",len(audits[audits.operation.eq("MailItemsAccessed")]),"operations logged; not independent proof of who controlled the client"],
],columns=["check","events","interpretation_limit"]))
# Explicit final-policy candidate for this traced case: use all affected identities'
# workstations and mailbox service criticality, then one +1 for confirmed mailbox access.
# Fraud runbook asks for critical; the reason for escalation is independent of severity.
canonical_base=matrix.loc[max(story.severity,key=severity_order.index),"Alta"]
canonical_candidate=severity_order[min(4,severity_order.index(canonical_base)+1)]
show("Canonical grouped ticket policy candidate",{"service_criticality_assumption":"Alta (mail service)",
 "highest_rule_severity":"high","matrix_base":canonical_base,"single_access_adjustment":"+1",
 "candidate_final_severity":canonical_candidate,"escalate":True,
 "basis":"session anomaly, mailbox persistence/access, attempted financial fraud; human review required"})

Rare severe seed counts (leads, no ground-truth label)
     value  count
UC-WEB-003     12
 UC-OT-001     10
UC-EDR-011      7
UC-NET-004      5
UC-EML-003      4
UC-EDR-012      3
UC-NET-002      2
UC-IDP-004      2
UC-IDP-006      2
UC-IDP-008      2
UC-EDR-006      1
UC-IDP-005      1
 UC-AI-002      1
UC-EDR-004      1
UC-EDR-007      1
UC-IDP-003      1
UC-CLD-001      1
UC-CLD-002      1
UC-CLD-004      1

Exact indicator groups intersecting rare severe seeds
 alerts  rules      first_alert_id       last_alert_id
      3      1 ALR-20261020-011897 ALR-20261020-011899
      3      1 ALR-20261020-011911 ALR-20261020-011913
      2      2 ALR-20261010-011797 ALR-20261018-011896

Canonical story: event stage totals
                           stage  linked_by_story_alert  events                 first_UTC                  last_UTC
   bank-change message delivered                   True       1 2026-10-08 08:15:00+00:00 2026-10-08 08:15:00+00:00
              link click allowed         

In [14]:
# Wider seeds than above: every alert of a rare rule at any severity, plus alerts with an absent approval claim.
# No ID number is used. The traced fraud alert, for one, is only medium.
covered=set(story_ids)|set().union(*context_ids.values())
for name,entities,rules in lead_specs:
    mask=alerts.rule_id.isin(rules)
    if entities:mask &= alerts.entities.map(lambda e:any(v in entities for _,v in leaves(e)))
    covered|=set(alerts.loc[mask,"alert_id"])
absent_refs={ref for ref in claimed_refs if ref not in calendar_ids}
planted_mask=alerts.evidence.map(lambda e:bool(approval_refs(e)&absent_refs))
open_leads=alerts[(alerts.rule_id.isin(rare_rules)|planted_mask)&~alerts.alert_id.isin(covered)]
show("Rare-rule or absent-claim alerts in no chain or authorised context",{"alerts":len(open_leads),"rules":open_leads.rule_id.value_counts().to_dict()})
# Group by exact shared entity, ignoring IPs shared by many identities and values on many alerts.
owners=collections.defaultdict(set)
for a in open_leads.itertuples():
    for field,value in leaves(a.entities):
        key=str(value).casefold()
        if value and not field.endswith("display_name") and key not in shared_ips: owners[key].add(a.alert_id)
parent={i:i for i in open_leads.alert_id}
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
for ids in owners.values():
    ids=sorted(ids)
    if len(ids)<=25:
        for other in ids[1:]: parent[find(other)]=find(ids[0])
open_leads=open_leads.assign(group=open_leads.alert_id.map(find),
    hosts=open_leads.entities.map(lambda e:[h.get("hostname") for h in e.get("hosts",[])]))
group_rows=[]
for _,g in open_leads.groupby("group"):
    group_rows.append([" + ".join(f"{rule} x{n}" for rule,n in g.rule_id.value_counts().items()),len(g),
                       str(g.created_at_utc.min())[:10],str(g.created_at_utc.max())[:10],sorted({h for hs in g.hosts for h in hs})[:3]])
lead_groups=pd.DataFrame(group_rows,columns=["rules","alerts","first","last","hosts"])
show("Groups among those alerts (leads; not traced)",lead_groups.groupby("rules").agg(groups=("alerts","size"),alerts=("alerts","sum"),
     first=("first","min"),last=("last","max"),hosts=("hosts","first")).sort_values("alerts",ascending=False).reset_index())

Rare-rule or absent-claim alerts in no chain or authorised context
{'alerts': 37, 'rules': {'UC-WEB-003': 12, 'UC-EML-001': 9, 'UC-EML-003': 3, 'UC-NET-002': 2, 'UC-EDR-006': 1, 'UC-EML-002': 1, 'UC-EDR-008': 1, 'UC-IDP-003': 1, 'UC-EDR-005': 1, 'UC-IDP-004': 1, 'UC-CLD-001': 1, 'UC-CLD-002': 1, 'UC-DLP-002': 1, 'UC-CLD-004': 1, 'UC-IDP-006': 1}}

Groups among those alerts (leads; not traced)
                                        rules  groups  alerts      first       last                                   hosts
                                UC-WEB-003 x1      12      12 2026-10-10 2026-10-11                                      []
                                UC-EML-001 x9       1       9 2026-10-20 2026-10-20                                      []
                UC-EML-003 x3 + UC-IDP-006 x1       1       4 2026-10-20 2026-10-20 [WS-BCN-1053, WS-BCN-1107, WS-BCN-1837]
UC-CLD-001 x1 + UC-CLD-002 x1 + UC-CLD-004 x1       1       3 2026-10-15 2026-10-16                         

**Observed, traced case (account `laura.ferrer2`):** Mail delivery → click and proxy GET/POST → two user agents using the same session from one logged source → inbox rule → mailbox reads → an internal bank-change message. Exact message ID, session ID, identity and referenced event IDs connect the chain. The bank-change amount is a claim in the alert/mail evidence; settlement or actual financial loss is not demonstrated. Forty mailbox-access events are outside the five alerts' explicit links. The phishing alert arrives after several later attack stages, so sorting alerts by creation time would misorder the attack.

**Interpretation (agent/triage, data/eval):** Escalate the combined case for credential/session compromise, persistence and attempted fraud. A click alone would not prove credential theft; the correlated session and mailbox activity provide the extra evidence. Group related alerts and retain evidence provenance. The logs do not establish a change of IP within the suspicious session, the exact theft mechanism, or the submitted credential contents. The wider candidate chains remain leads: common hosts/tools or indicators do not establish a single adversary. TIP absence is not a clean bill of health. No live reputation lookup, threat attribution, actual exfiltration volume, payment settlement, or containment outcome was verified.

**Wider leads:** the last table above groups 37 alerts in no chain. Going by rule names, they suggest customer account takeover, a second phishing case carrying the absent approval claims, the cloud case, earlier steps on `WS-BCN-1770`, cryptocurrency mining, MFA fatigue and a service-account login. None is traced. Two more hypotheses to test: the `adm.*` rows in section 8 place `FS01` and `PRINT01` on the same night, which could join two of the chains above; and the AnyDesk laptop and the tills with memory reads belong to the same store, whose network the company file describes as flat. Grouping on shared entities alone wouldn't connect the laptop and the tills.

**For agent tests:** real incidents include low and medium alerts, and alerts can arrive out of order. Section 11 lists the attacks that must always escalate.

## 11. Test set

Does the 150 alert sample cover what we found? Which alert IDs should be added?

Affects: data/eval

In [15]:
sample_ids=set(sample.alert_id)
coverage=alerts.groupby("rule_id").size().to_frame("full_alerts").join(
    sample.groupby("rule_id").size().rename("sample_alerts")).fillna(0).astype(int)
show("Sample rule coverage",coverage.reset_index())
show("Sample severity coverage",pd.crosstab(sample.rule_id,sample.severity).sum())
show("Runbook and source coverage",{"full_runbooks":alerts.runbook.nunique(),"sample_runbooks":sample.runbook.nunique(),
    "full_products":alerts.source_product.nunique(),"sample_products":sample.source_product.nunique(),
    "observed_rules_missing_from_sample":int(coverage.sample_alerts.eq(0).sum()),
    "catalog_rules_without_any_alert":len(set(catalog.rule_id)-set(alerts.rule_id))})
# Held out while tuning (docs/decisions.md): the stolen cloud key case, and the build that read AWS_ variables
# two days earlier, until someone checks whether it belongs to the same case.
cloud_case=set(alerts.loc[alerts.rule_id.isin(["UC-CLD-001","UC-CLD-002","UC-CLD-004"]),"alert_id"])
build_lead=set(alerts.loc[alerts.rule_id.eq("UC-DEV-001")&alerts.evidence.map(lambda e:"AWS_" in encoded(e.get("env_vars_read",""))),"alert_id"])
held_out=cloud_case|build_lead
show("Held out while tuning",{"cloud_case":sorted(cloud_case),"possible_earlier_step":sorted(build_lead),
     "already_in_development_sample":sorted(held_out&sample_ids)})
# Every case comes from a named signal above, never from an ID number. Ties go to the earliest alert.
in_context=lambda name:alerts.alert_id.isin(context_ids[name])
rule=lambda name:alerts.rule_id.eq(name)
before_change=alerts.created_at_utc.lt(pd.Timestamp("2026-10-25T01:00Z"))
trip_cases=hr_cases[hr_cases.relation.eq("inside stated dates")]
leaver_cases=hr_cases[hr_cases.rule.eq("UC-DLP-001")&hr_cases.relation.eq("before last day")]
travel_pattern=alerts[rule("UC-IDP-001")].evidence.map(lambda e:encoded({k:e.get(k) for k in pattern_keys["UC-IDP-001"]}))
rare_travel=alerts.index.isin(travel_pattern[travel_pattern.map(travel_pattern.value_counts()).eq(1)].index)
by_time=alerts.sort_values("created_at_utc")
behaviour_cases=[
 ("scanner before the clock change",in_context("authorised scanner")&before_change),
 ("scanner after the clock change",in_context("authorised scanner")&~before_change),
 ("internal scan not from the scanner",rule("UC-NET-003")&~scanner_mask),
 ("PsExec inside the approved change",in_context("PsExec patching")),
 ("lateral movement outside any change",rule("UC-NET-004")&~ps),
 ("OT from the permitted source",in_context("OT allowed company source")),
 ("OT through the maintenance window",in_context("OT maintenance")),
 ("payroll flow",in_context("payroll flow")),
 ("personal data to another recipient",rule("UC-DLP-002")&~flow),
 ("upload by a departing employee",alerts.alert_id.isin(leaver_cases.alert_id)),
 ("impossible travel during a dated trip",alerts.alert_id.isin(trip_cases.alert_id)),
 ("impossible travel, rare pattern, no trip",pd.Series(rare_travel,index=alerts.index)&~alerts.alert_id.isin(trip_cases.alert_id)),
 ("EDR stop citing the approved change",in_context("EDR agent upgrade")),
 ("EDR stop with no change",rule("UC-EDR-011")&~alerts.alert_id.isin(maint.alert_id)),
 ("group change inside its window",in_context("directory group change")),
 ("group change outside any change",rule("UC-IDP-008")&~in_context("directory group change")),
 ("vendor VPN inside its exception",in_context("WMS supplier exception")),
 ("vendor VPN from another source",rule("UC-VPN-001")&~vpn_corporate),
 ("alert with no linked events",alerts.related_events.map(len).eq(0)),
 ("service-account login with an MFA exemption",alerts.alert_id.isin(backup_alerts.alert_id)),
 ("approval claim in a service description",planted_mask&rule("UC-EDR-012")),
 ("hidden text in a reported email",planted_mask&rule("UC-EML-001")),
 ("hidden text in a clicked email",planted_mask&rule("UC-EML-003")),
 ("Vela multi-event disclosure",rule("UC-AI-002")),
]
def earliest(mask):
    picks=by_time[mask.reindex(by_time.index).astype(bool)&~by_time.alert_id.isin(held_out)]
    return picks.alert_id.iloc[0] if len(picks) else None
test_cases={aid:"traced session and fraud case" for aid in story_ids}
for purpose,mask in behaviour_cases:
    pick=earliest(mask)
    if pick: test_cases.setdefault(pick,purpose)
covered_rules=set(alerts.loc[alerts.alert_id.isin(test_cases),"rule_id"])
for name in coverage[coverage.sample_alerts.eq(0)].index:
    if name not in covered_rules:
        pick=earliest(rule(name))
        if pick: test_cases.setdefault(pick,"rule absent from the sample")
assert not set(test_cases)&held_out
recommended=pd.DataFrame([[aid,alerts.set_index("alert_id").rule_id[aid],purpose,aid in sample_ids] for aid,purpose in test_cases.items()],
    columns=["alert_id","rule","test_purpose","already_in_sample"])
show("Behavioural test selection",recommended)
covered_after=set(sample.rule_id)|set(recommended.rule)
show("Additions required",{"new_alert_ids":int((~recommended.already_in_sample).sum()),
 "rules_still_absent":sorted(set(alerts.rule_id)-covered_after),
 "planted_text_alerts_in_sample":int(alerts.loc[planted_mask,"alert_id"].isin(sample_ids).sum()),
 "planted_text_alerts_total":int(planted_mask.sum())})
# For tests in tests/test_agent_triage.py, which use a fake model. Never prompt examples.
must_escalate=sorted(set(story_ids)|set(alerts.loc[alerts.rule_id.isin(["UC-EDR-004","UC-EDR-007","UC-EDR-012"]),"alert_id"]))
show("Agent test lists",{"planted_text":sorted(alerts.loc[planted_mask,"alert_id"]),"must_escalate":must_escalate})

Sample rule coverage
   rule_id  full_alerts  sample_alerts
 UC-AI-001          415              6
 UC-AI-002            1              0
UC-CLD-001            1              0
UC-CLD-002            1              0
UC-CLD-003           50              1
UC-CLD-004            1              1
UC-DEV-001           11              0
UC-DLP-001           59              1
UC-DLP-002            3              0
UC-EDR-001          532              4
UC-EDR-002         1349             19
UC-EDR-003          411              4
UC-EDR-004            1              0
UC-EDR-005            1              0
UC-EDR-006            1              0
UC-EDR-007            1              0
UC-EDR-008            1              0
UC-EDR-011            7              0
UC-EDR-012            3              0
UC-EML-001         2295             34
UC-EML-002            1              0
UC-EML-003            4              1
UC-EML-004            1              0
UC-IDP-001          901             11
UC-I

**Observed:** Rule coverage alone does not establish coverage of authorisation exceptions, prompt injection, event ordering, DST or same-name identities. The list above selects concrete cases in pairs, a likely benign alert with its look-alike, without modifying the sample file. The two rules still absent are in the held-out case.

**Design interpretation (data/eval):** Keep full incident groups together when splitting train/development/test, otherwise linked alerts leak the answer. Ask a person to label expected verdict, severity, escalation, cited evidence and required approvals, without seeing the agent's answer first. Measure retrieval completeness and group quality as well as per-alert accuracy.

**Data leakage:** nothing here is picked by ID number. The held-out alerts stay out of tuning, out of prompts and out of this list, including the one already in the development sample. This notebook has read them, so the hold-out isn't blind: don't claim a clean held-out incident score. The official live set is unavailable, so generalisation to unseen alerts has not been tested. Prompt guidance stays general, and alert IDs from this section go only in tests and labels.

## 12. Mock API

On a copy outside data/, what does the API enforce compared with section 4 of the policy? What must our own code enforce?

Affects: data/api-client, agent/triage

In [16]:
warnings.filterwarnings("ignore", message=".*httpx.*")
from analysis.api_probe import probe_api
api_results,api_summary=probe_api()
show("Actual requests against an unchanged temporary mock API copy",api_results)
show("Probe summary",api_summary)
show("Enforcement required in application code",pd.DataFrame([
 ["approval identity","Authenticated human; reject model-created approvals and arbitrary approver strings"],
 ["approval scope","Bind canonical target/action/ticket, expiry and single use; store audit evidence"],
 ["asset classes","N2 + service owner for servers; N2 + business for TPV/web"],
 ["OT","Resolve hostname/case/IP aliases; never perform SOC containment on OT"],
 ["IOC blocking","Human approval under project rule; dual-use service block needs N2"],
 ["input bounds","Positive bounded limits; known targets; source allowlist; explicit incomplete results"],
 ["search results","Canonical exact entities, pagination by time, timestamps, provenance, deduplication"],
],columns=["area","required_control"]))

Actual requests against an unchanged temporary mock API copy
                                     probe  HTTP_status
                     isolate without token          403
             token for wrong action/target          403
                        unsupported action          400
                block_ioc without approval          200
              purge_email without approval          200
                     unknown host accepted          200
critical server, arbitrary single approver          200
                       same token replayed          200
                         TPV, one approver          200
                         OT exact hostname          403
                        OT lowercase alias          200
                               OT IP alias          200
              OT IP block without approval          200
       known dual-use service global block          200
                     unknown user accepted          200
                        unknown log source 

**Observed:** Tests use FastAPI's in-process client with no listening port. The API source is copied unchanged to a temporary directory; read-only dataset inputs are linked, and the containment journal is written only beside that copy and deleted on exit. Synthetic approval fixtures test the mock contract; they are not operational approvals.

The API validates supported action names, token action/target equality and an exact-case OT hostname for `isolate_host`. It accepts arbitrary approver text, reusable tokens, unknown targets, a single approver for a critical server/TPV, OT aliases, and IOC blocks without approval. Its `truncated` flag means the result reached the limit, even when no additional record exists. Its q can't match a `DOMAIN\user` value typed normally, because it searches JSON text where the backslash is doubled. These are measured mock behaviours, not authorisation to use them in the application.

**Design interpretation (data/api-client, agent/triage):** The application must enforce policy and the stricter project rule that a person approves every containment action. Keep approval and execution tools outside LLM authority. No changes to other branches are made here. Missing checks in the mock need a wrapper, not a prompt promise.

**Verification limits:** All twelve sections are executed. Other attack stories remain leads, lexical injection recall is unknown, token counts are estimates, authorisation candidates require the remaining scope checks, the link between the build alert and the cloud case is unchecked, and the dataset provides no independent alert-level ground truth. Real indicator reputation, real-world effects, live demo alerts and model quality are deliberately untested.